# Methoden en Technieken 2025-2026 -- Blok 3

## Datapunt Opdracht 3a

In deze opdracht worden de volgende leeruitkomsten getoetst, relevante termen zijn **dik** gedrukt:
- A2: Je stelt voor een AI-oplossing juridische, ethische, organisatorische, **functionele en technische requirements** op.
- B1: Je **verkent en prepareert een dataset voor het trainen en testen van een AI-model en kan de voor- en nadelen van het gebruik van een bestaande dataset onderbouwen**, rekening houdend met technische en ethische randvoorwaarden.
- B2: Je **stelt op basis van requirements en data een geschikte architectuur voor een AI-oplossing op en selecteert daarvoor passende AI-technieken gebruik makend van bijvoorbeeld** **machine learning**, deep learning, kennisrepresentatie, computer vision en **natural language processing**.
- B3: Je **ontwikkelt een nieuw** of voorgetraind **AI-model volgens een iteratief en systematisch proces**.
- C2: **Je evalueert en beoordeelt de kwaliteit van een AI-model aan de hand van kwaliteitscriteria die in het vakgebied erkend worden** zoals robustness, **performance**, scalability, explainability, **model complexity** en resource demand.


## De opdracht

Onderstaande code leest de data van verschillende *ratings* in. Deze dataset is de **MovieTweetings**-dataset (ook naar verwezen in Les 4 van blok 3) waar het MovieGEEKs-voorbeeld gebruik van maakt. In de data staan de waarderingen (van 0 t/m 10) van gebruikers voor verschillende films en bijbehorende *timestamp*. Zie ook https://github.com/sidooms/MovieTweetings/tree/master voor een uitleg van de dataset.

De bedoeling is om een aanbevelings-systeem te bouwen dat voor elke willekeurige gebruiker in het systeem drie films aanbeveelt. Probeer de aanbeveling zo persoonlijk mogelijk te maken.
* Kies een model en verantwoord deze keuze.
* Besluit hoe je het model beoordeelt (datasplitsing en maatstaf) en verantwoord deze keuze.
* Evalueer het model.
* Geef concrete suggesties om het model te verbeteren. Je hoeft deze verbeteringen niet uit te voeren.
* Bespreek voor- en nadelen van het model dat je hebt gemaakt.

In [36]:
# %pip install pandas surprise numpy==1.26.4 scikit-learn

In [ ]:
import os
# import torch

# Environment configuration
# os.environ["KERAS_BACKEND"] = "torch"

import pandas as pd
import numpy as np
import math
import warnings

from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import GridSearchCV as SurpriseGridSearchCV
from surprise.model_selection import KFold as SurpriseKFold
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
np.random.seed(42)

# Hardware check
# if torch.cuda.is_available():
#     print('Notebook gebruikt windows GPU')
# elif torch.mps.is_available():
#     print('Notebook gebruikt apple GPU')
# else:
#     print('Notebook gebruikt CPU')


## Data laden

Onderstaande code leest de data van verschillende *ratings* in. Deze dataset is de **MovieTweetings**-dataset (ook naar verwezen in Les 4 van blok 3) waar het MovieGEEKs-voorbeeld gebruik van maakt. In de data staan de waarderingen (van 0 t/m 10) van gebruikers voor verschillende films en bijbehorende *timestamp*. Zie ook https://github.com/sidooms/MovieTweetings/tree/master voor een uitleg van de dataset.

In [38]:
ratings = pd.read_csv(
    'https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/ratings.dat',
    delimiter='::', engine='python', header=None,
    names=['user_id', 'movie_id', 'rating', 'timestamp']
)
print(f"Loaded {len(ratings):,} ratings")

Loaded 921,398 ratings


### Movie metadata

Naast de ratings laad ik ook de film metadata (titel, jaar, genres) in. Deze wil ik gebruiken om een content-based component te maken voor een hybride model en om de uiteindelijke aanbevelingen beter interpreteerbaar te presenteren.

In [ ]:
items_raw = pd.read_csv(
    'https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/movies.dat',
    delimiter='::', engine='python', header=None,
    names=['movie_id', 'title_raw', 'genres_raw'],
    encoding='utf-8'
)

# Haal het jaar uit de titel, bijv. "Toy Story (1995)" → title="Toy Story", year=1995
items_raw['year'] = items_raw['title_raw'].str.extract(r'\((\d{4})\)').astype(float)
items_raw['title'] = items_raw['title_raw'].str.replace(r'\s*\(\d{4}\)\s*$', '', regex=True)

# Splits genres op in een lijst en vang lege waarden op
items_raw['genres'] = (
    items_raw['genres_raw'].fillna('').str.split('|', regex=False)
    .apply(lambda g: [x for x in g if x])
)

# Zorg voor unieke film-metadata per movie_id
items = (
    items_raw[['movie_id', 'title', 'year', 'genres']]
    .drop_duplicates(subset=['movie_id'], keep='last')
    .reset_index(drop=True)
)
print(f"Loaded {len(items):,} unique movies")

# Combineer ratings met filminformatie
df = ratings.merge(items, on='movie_id', how='inner')

# Zorg dat rating numeriek is en binnen 0–10 valt
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df[(df['rating'] >= 0) & (df['rating'] <= 10)].dropna(subset=['rating', 'genres'])

Loaded 38,013 unique movies


## Eerste inspectie van de data

Een snelle controle op het aantal observaties, kolomtypen en ontbrekende waarden vóór verdere verkenning. Gedetailleerde exploratie volgt na de datasplit, uitsluitend op de trainingsset.

In [ ]:
print(f"Aantal observaties: {len(df):,}")
print(f"Aantal kolommen:    {df.shape[1]}")
print(f"Unieke gebruikers:  {df['user_id'].nunique():,}")
print(f"Unieke films:       {df['movie_id'].nunique():,}")
print()
df.info()

Aantal observaties: 921,398
Aantal kolommen:    7
Unieke gebruikers:  71,707
Unieke films:       38,013

<class 'pandas.DataFrame'>
RangeIndex: 921398 entries, 0 to 921397
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   user_id    921398 non-null  int64  
 1   movie_id   921398 non-null  int64  
 2   rating     921398 non-null  int64  
 3   timestamp  921398 non-null  int64  
 4   title      921398 non-null  str    
 5   year       921398 non-null  float64
 6   genres     921398 non-null  object 
dtypes: float64(1), int64(4), object(1), str(1)
memory usage: 49.2+ MB


## Voor- en nadelen van het gebruik van de dataset

### Voordelen

- De dataset is publiek beschikbaar en open voor onderzoek, waardoor experimenten reproduceerbaar zijn en resultaten eenvoudig met andere studies kunnen worden vergeleken.
- De dataset wordt veel gebruikt in recommender system onderzoek, waardoor prestaties van modellen goed te vergelijken zijn met eerder werk.
- De ratings zijn afkomstig van echte gebruikersinteracties op sociale media en zijn dus niet kunstmatig gegenereerd.
- De datastructuur is eenvoudig en vergelijkbaar met bekende datasets zoals MovieLens (users, items en ratings), waardoor deze makkelijk te importeren en te gebruiken is.
- De ratings gebruiken een schaal van 0 tot10, waardoor meer nuance mogelijk is dan bij bijvoorbeeld een 5-sterren systeem.
- De dataset bevat timestamps voor elke rating waardoor tijdsgebaseerde analyses of temporele train/test splits mogelijk zijn.
- Films worden geïdentificeerd met IMDb identifiers, waardoor eenvoudig extra metadata kan worden toegevoegd uit externe bronnen.
- De dataset bevat basis metadata zoals filmgenres, waardoor ook content-based of hybride recommender systemen onderzocht kunnen worden

### Nadelen

- De dataset wordt niet meer actief geüpdatet en bevat sinds 2021 geen nieuwe ratings meer, waardoor deze minder representatief is voor recente films en gebruikersvoorkeuren.
- De dataset is sterk sparse, omdat veel gebruikers slechts een klein aantal films beoordelen en veel films weinig ratings hebben.
- Door de sparsity ontstaan veel cold-start gebruikers en films, wat het moeilijker maakt om betrouwbare aanbevelingen te genereren.
- De dataset bevat alleen ratings van gebruikers die hun IMDb beoordeling via Twitter delen, waardoor de gebruikerspopulatie minder representatief is voor filmkijkers in het algemeen.
- De dataset bevat weinig gebruikersmetadata zoals leeftijd, locatie of geslacht, wat recommendations op basis van user similarity moeilijk maakt.
- De inhoudelijke metadata van films is beperkt tot basisinformatie zoals titel en genre, waardoor extra data nodig kan zijn voor uitgebreidere content-based modellen.
- De ratings worden automatisch uit tweets geëxtraheerd, waardoor het mogelijk is dat sommige ratings verkeerd worden geïnterpreteerd.
- De dataset bevat Twitter user IDs, wat als persoonlijke data kan worden beschouwd omdat je user names er mee kan opvragen.
- Omdat de dataset bekend is en veel gebruikt wordt in onderzoek, kan voorafgaande kennis over eigenschappen zoals sparsity of ratingdistributies een vorm van data leakage veroorzaken tijdens exploratie.
- De dataset heeft een populariteitsbias, omdat films waar vaker over getweet wordt waarschijnlijk ook vaker ratings krijgen.

## Data Preparation

### Datasetopschoning

Voordat het recommender model wordt getraind wordt de dataset eerst opgeschoond. Gebruikers met zeer weinig interacties worden gefilterd omdat zij onvoldoende informatie bieden om betrouwbare aanbevelingen te genereren [1, p.229]. Datasets voor recommender systemen vereisen doorgaans enige opschoning, omdat gebruikers met slechts enkele ratings weinig bijdragen aan het leerproces van het model [1, p.236].

In dit project worden **gebruikers met minder dan 5 ratings verwijderd** en **films met minder dan 3 ratings gefilterd**, omdat zeer zeldzame films de sparsity vergroten.

### Minimum ratings voor collaborative filtering

Collaborative filtering vereist voldoende interactiedata om gelijkenissen tussen gebruikers of items te berekenen. Gebruikers met slechts één rating leveren geen bruikbare informatie voor dit type algoritme [1, p.229]. Door een minimum te vereisen ontstaat meer overlap tussen gebruikersinteracties, wat essentieel is voor betrouwbare similarityberekeningen [1, p.242].

### Chronologische ordening en datasplit

Per gebruiker worden ratings **chronologisch gesorteerd op timestamp**. Vervolgens wordt een **given-n protocol** toegepast: de eerste **5 ratings** gaan naar de trainingsset, de rest naar de testset.

In [ ]:
# Verwijder gebruikers met minder dan 5 ratings
user_counts = df.groupby('user_id').size()
valid_users = user_counts[user_counts >= 5].index
df = df[df['user_id'].isin(valid_users)].copy()

print(f"Na filteren gebruikers (< 5 ratings): {len(df):,} ratings  ({df['user_id'].nunique():,} users)")

# Verwijder films met minder dan 3 ratings
movie_counts = df.groupby('movie_id').size()
valid_movies = movie_counts[movie_counts >= 3].index
df = df[df['movie_id'].isin(valid_movies)].copy()

print(f"Na filteren films (< 3 ratings):      {len(df):,} ratings  ({df['user_id'].nunique():,} users, {df['movie_id'].nunique():,} movies)")

After removing users with < 5 ratings: 845,154 ratings  (23,805 users)
After removing movies with < 3 ratings: 819,491 ratings  (23,802 users, 16,384 movies)


## Data Split met Given-n Protocol (chronologisch)

### Train/test datasplitsing

De dataset wordt tijdsgebaseerd gesplitst zodat het model alleen leert van eerdere interacties en voorspellingen maakt voor latere interacties [1, p.233]. Een chronologische splitsing voorkomt **data leakage** en benadert beter de werkelijke toepassing van recommender systemen, waarin aanbevelingen altijd worden gedaan op basis van historisch gedrag. Dit is ook realistischer omdat iemands voorkeuren over tijd kunnen veranderen.

Daarnaast wordt een **per-gebruiker splitsing** toegepast: de eerste ratings van een gebruiker gaan naar training, latere naar test, zodat iedere gebruiker in de trainingsdata voorkomt [1, p.233–234].

### Given-n protocol

In dit project wordt een **given-n protocol** gebruikt met $n = 5$. Per gebruiker worden alle ratings chronologisch gesorteerd. De eerste **5 ratings** gaan naar de trainingsset, de rest naar de testset.

### Cold-start gebruikers

Gebruikers met minder dan **5 ratings in de trainingsset** worden als cold-start beschouwd en krijgen aanbevelingen via een populariteitsbaseline. In de praktijk is dit een vangnet: door de voorafgaande filtering (≥ 5 ratings) en het given-n protocol (eerste 5 naar train) valt vrijwel niemand in deze categorie.

In [43]:
GIVEN_N = 5

def given_n_split(data, n=GIVEN_N):
    """
    Chronologische split per gebruiker: de eerste n ratings (op timestamp)
    gaan naar train, de rest naar test.
    Gebruikers met <= n ratings gaan volledig naar train.
    """
    train_parts, test_parts = [], []
    for _, group in data.groupby('user_id'):
        sorted_group = group.sort_values('timestamp')
        train_parts.append(sorted_group.iloc[:n])
        if len(sorted_group) > n:
            test_parts.append(sorted_group.iloc[n:])

    train = pd.concat(train_parts).reset_index(drop=True)
    test = pd.concat(test_parts).reset_index(drop=True) if test_parts else pd.DataFrame(columns=data.columns)
    return train, test

train_df, test_df = given_n_split(df, n=GIVEN_N)

# Zorg dat elke testgebruiker ook in de trainingsdata voorkomt
train_user_set = set(train_df['user_id'].unique())
test_df = test_df[test_df['user_id'].isin(train_user_set)].copy()

print(f"Train: {len(train_df):,} ratings  ({train_df['user_id'].nunique():,} users)")
print(f"Test:  {len(test_df):,} ratings  ({test_df['user_id'].nunique():,} users)")
print(f"Alle testgebruikers in training: {set(test_df['user_id'].unique()).issubset(train_user_set)}")

# Identificeer cold-start gebruikers (< 5 training ratings)
COLD_THRESHOLD = 5
ratings_per_user = train_df.groupby('user_id').size().to_dict()
cold_users = {u for u, c in ratings_per_user.items() if c < COLD_THRESHOLD}
warm_users = {u for u, c in ratings_per_user.items() if c >= COLD_THRESHOLD}

print(f"\nCold-start gebruikers (< {COLD_THRESHOLD} training ratings): {len(cold_users):,}")
print(f"Warm gebruikers (>= {COLD_THRESHOLD} training ratings):      {len(warm_users):,}")

Train: 118,717 ratings  (23,802 users)
Test:  700,774 ratings  (21,205 users)
All test users appear in training: True

Cold users (< 20 training ratings): 23,802
Warm users (≥ 20 training ratings): 0


## Data Exploration

Nu de data is gesplitst, verkennen we de **trainingsset** om inzicht te krijgen in de verdeling van ratings, het aantal gebruikers en films. Alle exploratie vindt plaats op de trainingsdata om datalekkage te voorkomen.

In [52]:
print("=== Training Set Overview ===\n")
n_users_train = train_df['user_id'].nunique()
n_movies_train = train_df['movie_id'].nunique()
n_ratings_train = len(train_df)

print(f"Users:               {n_users_train:,}")
print(f"Movies:              {n_movies_train:,}")
print(f"Total ratings:       {n_ratings_train:,}")
print(f"Avg ratings / user:  {n_ratings_train / n_users_train:.2f}")
print(f"Rating range:        {train_df['rating'].min():.0f}–{train_df['rating'].max():.0f}")
print(f"Mean rating:         {train_df['rating'].mean():.2f}")

print("\n=== Test Set Overview ===\n")
print(f"Users:               {test_df['user_id'].nunique():,}")
print(f"Ratings:             {len(test_df):,}")

print("\n=== Rating Distribution (train) ===")
print(train_df['rating'].value_counts().sort_index().to_string())

print("\n=== Genres in de dataset ===")
unique_genres = sorted({
    genre
    for genres in train_df['genres'].dropna()
    for genre in (genres if isinstance(genres, list) else [genres])
})
print(f"Unique genres:       {len(unique_genres):,}")
print(f"Genres:              {', '.join(unique_genres)}")

print("\n=== Voorbeeld trainingsdata ===")
train_df.head(10)

=== Training Set Overview ===

Users:               23,802
Movies:              10,239
Total ratings:       118,717
Avg ratings / user:  4.99
Rating range:        0–10
Mean rating:         7.61

=== Test Set Overview ===

Users:               21,205
Ratings:             700,774

=== Rating Distribution (train) ===
rating
0        46
1      1462
2      1004
3      1629
4      2810
5      6934
6     12168
7     23417
8     29610
9     20484
10    19153

=== Sample of training data ===
Unique genres:       24
Genres:              Action, Adventure, Animation, Biography, Comedy, Crime, Documentary, Drama, Family, Fantasy, Film-Noir, History, Horror, Music, Musical, Mystery, News, Romance, Sci-Fi, Short, Sport, Thriller, War, Western


,user_id,movie_id,rating,timestamp,title,year,genres
0,3,10039344,5,1578603053,Countdown,2019.0,"[Horror, Thriller]"
1,3,6751668,9,1578955697,Gisaengchung,2019.0,[Drama]
2,3,358273,9,1579057827,Walk the Line,2005.0,"[Biography, Drama, Music, Romance]"
3,3,8579674,10,1579261830,1917,2019.0,"[Drama, War]"
4,3,7131622,8,1579559244,Once Upon a Time ...in Hollywood,2019.0,"[Comedy, Drama]"
5,4,2278871,8,1383419733,La vie d'Adèle,2013.0,"[Drama, Romance]"
6,4,2395417,8,1388170007,Still Life,2013.0,[Drama]
7,4,1800241,7,1388955438,American Hustle,2013.0,"[Crime, Drama]"
8,4,790636,8,1391207279,Dallas Buyers Club,2013.0,"[Biography, Drama]"
9,4,3344922,8,1422652427,Hungry Hearts,2014.0,"[Drama, Romance, Thriller]"


## Modelvereisten en Architectuur

### Modelkeuze

Het aanbevelingssysteem gebruikt een **hybride recommender model** dat *collaborative filtering* combineert met *content-based filtering*. Collaborative filtering leert patronen uit gebruikers–item interacties voor sterke personalisatie. Content-based filtering gebruikt filmkenmerken en blijft bruikbaar bij weinig interactiedata. Door beide te combineren worden zowel gebruikersvoorkeuren als filmkenmerken benut en wordt het cold-start probleem deels verminderd.

### Functionele en technische requirements

1. Het systeem genereert voor **elke gebruiker drie gepersonaliseerde film-aanbevelingen** op basis van historische ratings, gerangschikt op voorspelde voorkeursscore.

2. De architectuur is **hybride**: collaborative filtering en content-based filtering worden gecombineerd.

3. Het collaborative filtering component gebruikt **matrixfactorisatie via SVD** (Surprise-bibliotheek) om latente gebruikers- en itemfactoren te leren.

4. Het SVD-model modelleert **gebruikers- en itembias** ($b_u$, $b_i$) zodat systematische verschillen in beoordelingsgedrag automatisch worden gecorrigeerd.

5. **Filmgenres worden gerepresenteerd als one-hot vectors** voor het berekenen van inhoudelijke overeenkomsten.

6. Overeenkomsten worden berekend met **cosine similarity**.

7. Het **gebruikersprofiel** wordt opgebouwd uit het gemiddelde van de genrevectoren van films met rating ≥ 7.

8. De aanbevelingsscore wordt berekend als:
   `final_score = 0.8 × collaborative_score + 0.2 × content_score`

9. **Cold-start afhandeling:** gebruikers met minder dan 5 training ratings krijgen aanbevelingen via een populariteitsbaseline. Films met minder dan 5 training ratings krijgen als collaborative score het gebruikersgemiddelde.

10. De eerste twee aanbevelingen worden gekozen op hoogste hybride score, de **derde uit een ander genrecluster** voor diversiteit.

### Hyperparameteroptimalisatie

De SVD-hyperparameters (`n_factors`, `n_epochs`, `lr_all`, `reg_all`) worden geoptimaliseerd met **grid search** [1, p.242]. Dit is haalbaar omdat het model klein is en het aantal parameters beperkt.

### Evaluatie

De kwaliteit wordt gemeten met **RMSE** (bestraft grote voorspelfouten) en **Precision@3** (aandeel relevante films in de top 3). Het hybride model wordt vergeleken met een **populariteitsbaseline**:

`popularity_score = average_rating × log(1 + number_of_ratings)`

Het model voldoet aan de doelen wanneer het een **lagere RMSE en hogere Precision@3** behaalt dan de baseline.

## Prestatiemaatstaven

### RMSE

**Root Mean Squared Error (RMSE)** meet het verschil tussen voorspelde en werkelijke ratings. Grote fouten worden zwaarder bestraft [1, p.224].

### Precision@3

**Precision@3** meet welk aandeel van de drie aanbevolen films daadwerkelijk relevant is (rating ≥ 7 in de testset). Lage precisie is normaal bij recommender systemen vanwege de grote itemset [1, p.239].

### Cross-validation

**5-fold cross-validation** vermindert de afhankelijkheid van één specifieke split en levert een betrouwbaardere RMSE-schatting [1, p.236].

### Baseline-vergelijking

Het hybride model wordt vergeleken met een **populariteitsbaseline** die films aanbeveelt op basis van:

$$\text{popularity\_score} = \overline{r} \cdot \ln(1 + n)$$

Hierdoor kan objectief worden vastgesteld of personalisatie waarde toevoegt.

## Base Model — Popularity Recommender

Een populariteitsbaseline dient als referentiepunt [1, p.239]. Per film wordt berekend:

$$\text{popularity\_score} = \overline{r} \cdot \ln(1 + n)$$

waarbij $\overline{r}$ de gemiddelde rating is en $n$ het aantal ratings. Deze baseline wordt ook gebruikt voor eventuele cold-start gebruikers.

In [53]:
# Popularity baseline: score = gemiddelde rating × log(1 + aantal ratings)
movie_stats = train_df.groupby('movie_id').agg(
    num_ratings=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

movie_stats['popularity_score'] = (
    movie_stats['avg_rating'] * np.log1p(movie_stats['num_ratings'])
)
movie_stats = movie_stats.merge(
    items[['movie_id', 'title', 'genres']], on='movie_id', how='left'
).sort_values('popularity_score', ascending=False).reset_index(drop=True)

# Snelle lookup-dicts voor later gebruik
popularity_dict = movie_stats.set_index('movie_id')['popularity_score'].to_dict()
movie_avg_dict = movie_stats.set_index('movie_id')['avg_rating'].to_dict()

print("Top-10 populairste films (baseline):\n")
print(
    movie_stats[['title', 'avg_rating', 'num_ratings', 'popularity_score']]
    .head(10)
    .to_string(index=False)
)

Top-10 most popular movies (baseline):

                              title  avg_rating  num_ratings  popularity_score
                               1917    8.573209          963         58.907300
                       Interstellar    9.133721          516         57.067880
           The Shawshank Redemption    9.561688          308         54.820422
                   Django Unchained    8.645390          564         54.784330
                            Gravity    8.254808          624         53.142402
                              Joker    9.104167          336         52.987005
                       Gisaengchung    8.424242          528         52.828327
            The Wolf of Wall Street    8.527352          457         52.245972
                      Hacksaw Ridge    8.978593          327         52.013113
            Star Trek Into Darkness    8.235185          540         51.827473
                     Iron Man Three    7.709599          823         51.763563
            

## Content-Based Filtering (genre similarity)

1. One-hot encode de genres naar een binaire featurematrix.
2. Bouw een **gebruikersprofiel** door de genrevectoren van films met rating ≥ 7 te middelen.
3. Bereken **cosine similarity** tussen elk gebruikersprofiel en elke filmgenrevector.

In [46]:
# One-hot encode genres naar een binaire film × genre matrix
genre_encoder = MultiLabelBinarizer()
genre_matrix = pd.DataFrame(
    genre_encoder.fit_transform(items['genres']),
    columns=genre_encoder.classes_,
    index=items['movie_id']
)
genre_matrix = genre_matrix.groupby(level=0).max()

if '(no genres listed)' in genre_matrix.columns:
    genre_matrix.drop(columns=['(no genres listed)'], inplace=True)

all_genre_names = list(genre_matrix.columns)
print(f"Genre features ({len(all_genre_names)}): {', '.join(all_genre_names)}")

# Bouw gebruikersprofielen: gemiddelde genrevector van films met rating >= 7
LIKE_THRESHOLD = 7
user_profiles = {}
for user_id, group in train_df.groupby('user_id'):
    liked_movies = group[group['rating'] >= LIKE_THRESHOLD]['movie_id']
    genre_vectors = genre_matrix.loc[genre_matrix.index.intersection(liked_movies)]

    # Fallback: gebruik alle beoordeelde films als er geen 'liked' films zijn
    if len(genre_vectors) == 0:
        genre_vectors = genre_matrix.loc[genre_matrix.index.intersection(group['movie_id'])]

    if len(genre_vectors) > 0:
        user_profiles[user_id] = genre_vectors.mean().values
    else:
        user_profiles[user_id] = np.zeros(genre_matrix.shape[1])

# Bereken cosine similarity matrix (gebruikers × films)
user_ids_ordered = list(user_profiles.keys())
user_profile_matrix = np.array([user_profiles[u] for u in user_ids_ordered])
movie_ids_ordered = genre_matrix.index.tolist()

content_similarity = cosine_similarity(user_profile_matrix, genre_matrix.values)

user_to_index = {u: i for i, u in enumerate(user_ids_ordered)}
movie_to_index = {m: j for j, m in enumerate(movie_ids_ordered)}

def get_content_score(user_id, movie_id):
    """Geeft de cosine similarity tussen een gebruikersprofiel en een film."""
    ui = user_to_index.get(user_id)
    mi = movie_to_index.get(movie_id)
    if ui is None or mi is None:
        return 0.0
    return float(content_similarity[ui, mi])

print(f"Gebruikersprofielen gebouwd voor {len(user_profiles):,} users")
print(f"Content-similarity matrix: {content_similarity.shape}")

Genre features (28): Action, Adult, Adventure, Animation, Biography, Comedy, Crime, Documentary, Drama, Family, Fantasy, Film-Noir, Game-Show, History, Horror, Music, Musical, Mystery, News, Reality-TV, Romance, Sci-Fi, Short, Sport, Talk-Show, Thriller, War, Western
User profiles built for 23,802 users
Content-similarity matrix shape: (23802, 38013)


## Collaborative Filtering — SVD met Hyperparameter Tuning

Surprise's SVD gebruikt de formule $\hat{r}_{ui} = \mu + b_u + b_i + \mathbf{q}_i^\top \mathbf{p}_u$, waarbij $\mu$ het globale gemiddelde is, $b_u$ en $b_i$ geleerde biases, en $\mathbf{p}_u$, $\mathbf{q}_i$ latente factorvectoren. De biases vangen verschillen in beoordelingsgedrag op, waardoor expliciet centreren niet nodig is.

**Hyperparameter tuning:** Grid search over `n_factors` ∈ {50, 100, 150}, `n_epochs` ∈ {20, 30}, `lr_all` ∈ {0.002, 0.005, 0.01}, en `reg_all` ∈ {0.02, 0.05, 0.1} met 3-fold CV. De beste parameters worden gebruikt om het finale model te trainen op de volledige trainingsset.

In [47]:
# Gemiddelde rating per gebruiker (fallback voor onbekende films)
user_mean_rating = train_df.groupby('user_id')['rating'].mean().to_dict()

# Surprise dataset opbouwen
reader = Reader(rating_scale=(0, 10))
surprise_data = Dataset.load_from_df(
    train_df[['user_id', 'movie_id', 'rating']], reader
)

# Grid search voor SVD hyperparameters (3-fold CV)
param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs':  [20, 30],
    'lr_all':    [0.002, 0.005, 0.01],
    'reg_all':   [0.02, 0.05, 0.1],
}

print("Running SVD grid search (3-fold CV) — dit kan een paar minuten duren …")
gs = SurpriseGridSearchCV(SVD, param_grid, measures=['rmse'], cv=3,
                          refit=False, n_jobs=-1)
gs.fit(surprise_data)

best_params = gs.best_params['rmse']
print(f"\nBeste RMSE (CV): {gs.best_score['rmse']:.4f}")
print(f"Beste parameters: {best_params}")

# Train het finale SVD model met de beste parameters op de volledige trainingsset
trainset = surprise_data.build_full_trainset()

svd = SVD(
    n_factors=best_params['n_factors'],
    n_epochs=best_params['n_epochs'],
    lr_all=best_params['lr_all'],
    reg_all=best_params['reg_all'],
    random_state=42
)
svd.fit(trainset)

print(f"\nFinaal SVD model getraind — factors={svd.n_factors}, epochs={svd.n_epochs}")
print(f"Trainset: {trainset.n_users} users, {trainset.n_items} items, {trainset.n_ratings} ratings")

Running SVD grid search (3-fold CV) — this may take a few minutes …

Best RMSE (CV): 1.6187
Best params:    {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.01, 'reg_all': 0.1}

Final SVD model trained — factors=50, epochs=20
Internal trainset: 23802 users, 10239 items, 118717 ratings


## Hybride Scoring & Cold-Start Afhandeling

$$\text{final\_score} = 0.8 \times \text{collaborative\_score} + 0.2 \times \text{content\_score}$$

De collaborative score komt van het SVD-model (inclusief biases). De content score (cosine similarity, 0–1) wordt geschaald naar 0–10 zodat beide componenten vergelijkbaar zijn.

**Cold-start regels:**
- **Gebruikers** met < 5 training ratings → popularity fallback met genrediversiteit.
- **Films** met < 5 training ratings → gebruikersgemiddelde als collaborative score, zodat het content-signaal domineert.

In [48]:
# Hybride gewichten
COLLAB_WEIGHT = 0.8
CONTENT_WEIGHT = 0.2

# Handige lookups
ratings_per_movie = train_df.groupby('movie_id').size().to_dict()
watched_movies = train_df.groupby('user_id')['movie_id'].apply(set).to_dict()
all_movies = set(items['movie_id'].unique())
global_avg = train_df['rating'].mean()

def predict_svd_scores(user_id, movie_ids):
    """Bereken SVD-voorspellingen voor een lijst films."""
    n = len(movie_ids)
    fallback = user_mean_rating.get(user_id, global_avg)

    try:
        inner_uid = svd.trainset.to_inner_uid(user_id)
    except ValueError:
        return np.full(n, fallback)

    pu = svd.pu[inner_uid]
    bu = svd.bu[inner_uid]
    mu = svd.trainset.global_mean

    predictions = np.full(n, fallback)
    valid = np.zeros(n, dtype=bool)
    inner_iids = np.zeros(n, dtype=int)

    for i, mid in enumerate(movie_ids):
        try:
            inner_iids[i] = svd.trainset.to_inner_iid(mid)
            valid[i] = True
        except ValueError:
            pass

    if valid.any():
        qi_batch = svd.qi[inner_iids[valid]]
        bi_batch = svd.bi[inner_iids[valid]]
        predictions[valid] = mu + bu + bi_batch + qi_batch @ pu

    return np.clip(predictions, 0, 10)

print(f"Hybride gewichten: collab={COLLAB_WEIGHT}, content={CONTENT_WEIGHT}")
print(f"Cold-start drempel: gebruiker < {COLD_THRESHOLD} ratings → popularity fallback")
print(f"Cold-start film drempel: < 5 ratings → user mean ({global_avg:.2f} global avg)")

Hybrid weights: collab=0.8, content=0.2
Cold-start threshold: user < 20 ratings → popularity fallback
Cold-start movie threshold: < 5 ratings → user mean (7.61 global avg)


## Aanbevelingsfunctie met Diversiteit

Per gebruiker:
1. Bereken de hybride score voor alle onbeoordeelde films.
2. Selecteer de **top 2** op score.
3. Selecteer een **diverse 3e film** waarvan de genres niet een subset zijn van de eerste twee — dit voegt serendipity toe.

Cold-start gebruikers (< 5 training ratings) ontvangen populariteitsgebaseerde keuzes met afgedwongen genrediversiteit.

In [54]:
def recommend(user_id, n=3):
    """
    Geeft een DataFrame met de top-n aanbevolen films voor een gebruiker.
    Dwingt genrediversiteit af voor de derde aanbeveling.
    """
    rated = watched_movies.get(user_id, set())
    candidates = np.array(list(all_movies - rated))
    if len(candidates) == 0:
        return pd.DataFrame(columns=['rank', 'movie_id', 'title', 'genres', 'score'])

    is_cold = user_id in cold_users

    # Bereken scores voor alle kandidaten
    if is_cold:
        scores = np.array([popularity_dict.get(m, 0.0) for m in candidates])
    else:
        collab_scores = predict_svd_scores(user_id, candidates)
        ui = user_to_index.get(user_id)
        if ui is not None:
            cb_scores = np.array([
                content_similarity[ui, movie_to_index[m]] * 10.0
                if m in movie_to_index else 0.0
                for m in candidates
            ])
        else:
            cb_scores = np.zeros(len(candidates))
        scores = COLLAB_WEIGHT * collab_scores + CONTENT_WEIGHT * cb_scores

    # Sorteer op score (hoog → laag)
    order = np.argsort(-scores)
    sorted_candidates = candidates[order]
    sorted_scores = scores[order]

    if len(sorted_candidates) <= n or n <= 2:
        return format_recommendations(list(zip(sorted_candidates[:n], sorted_scores[:n])))

    if is_cold:
        return diverse_popular_picks(sorted_candidates, sorted_scores, n)

    # Top 2 op basis van hybride score
    top2 = [(sorted_candidates[0], sorted_scores[0]),
            (sorted_candidates[1], sorted_scores[1])]

    # Genres die al in top 2 zitten
    top2_genres = set()
    for mid, _ in top2:
        if mid in movie_to_index:
            row = genre_matrix.loc[mid]
            top2_genres.update(row[row == 1].index)

    # Diverse 3e keuze: eerste kandidaat met genres die NIET een subset zijn van top-2
    diverse_pick = None
    for i in range(2, min(80, len(sorted_candidates))):
        mid = sorted_candidates[i]
        if mid in movie_to_index:
            movie_genres = set(genre_matrix.loc[mid][genre_matrix.loc[mid] == 1].index)
            if movie_genres and not movie_genres.issubset(top2_genres):
                diverse_pick = (mid, sorted_scores[i])
                break

    if diverse_pick is None:
        diverse_pick = (sorted_candidates[2], sorted_scores[2])

    return format_recommendations([top2[0], top2[1], diverse_pick])


def diverse_popular_picks(sorted_candidates, sorted_scores, n):
    """Kies top-n populaire films zonder genre-overlap tussen de keuzes."""
    picked, seen_genres = [], set()
    for mid, sc in zip(sorted_candidates, sorted_scores):
        if mid in movie_to_index:
            movie_genres = set(genre_matrix.loc[mid][genre_matrix.loc[mid] == 1].index)
            if movie_genres.isdisjoint(seen_genres) or len(picked) == 0:
                picked.append((mid, sc))
                seen_genres.update(movie_genres)
        if len(picked) >= n:
            break

    # Vul resterende plekken aan vanuit top als nodig
    if len(picked) < n:
        already_picked = {p[0] for p in picked}
        for mid, sc in zip(sorted_candidates, sorted_scores):
            if mid not in already_picked:
                picked.append((mid, sc))
            if len(picked) >= n:
                break
    return format_recommendations(picked[:n])


def format_recommendations(picked):
    """Zet een lijst van (movie_id, score) tuples om naar een presentatie-DataFrame."""
    rows = []
    for rank, (mid, sc) in enumerate(picked, 1):
        info = items[items['movie_id'] == mid]
        title = info['title'].values[0] if len(info) else f"Movie {mid}"
        genres = ', '.join(info['genres'].values[0]) if len(info) and isinstance(info['genres'].values[0], list) else ''
        rows.append({
            'rank': rank, 'movie_id': int(mid),
            'title': title, 'genres': genres,
            'score': round(float(sc), 3)
        })
    return pd.DataFrame(rows)


print("recommend() functie gereed.")

recommend() function ready.


## Evaluatie

Het **hybride model** en de **popularity baseline** worden vergeleken op RMSE en Precision@3. Daarnaast wordt 5-fold cross-validation uitgevoerd op het SVD-component voor een robuustere RMSE-schatting. Precision@3 wordt berekend op 500+ testgebruikers om de variantie te verlagen.

In [55]:
# 1) RMSE — hybride model
def compute_rmse_hybrid(test_data):
    user_ids = test_data['user_id'].values
    movie_ids = test_data['movie_id'].values
    actuals = test_data['rating'].values
    preds = np.empty(len(actuals))

    mu = svd.trainset.global_mean
    for i in range(len(actuals)):
        uid, mid = user_ids[i], movie_ids[i]
        fallback = user_mean_rating.get(uid, global_avg)

        if uid in cold_users:
            preds[i] = movie_avg_dict.get(mid, global_avg)
            continue

        if ratings_per_movie.get(mid, 0) < 5:
            collab = fallback
        else:
            try:
                inner_uid = svd.trainset.to_inner_uid(uid)
                inner_iid = svd.trainset.to_inner_iid(mid)
                collab = np.clip(
                    mu + svd.bu[inner_uid] + svd.bi[inner_iid] + np.dot(svd.pu[inner_uid], svd.qi[inner_iid]),
                    0, 10
                )
            except ValueError:
                collab = fallback

        cb = get_content_score(uid, mid) * 10.0
        preds[i] = COLLAB_WEIGHT * collab + CONTENT_WEIGHT * cb

    return np.sqrt(mean_squared_error(actuals, np.clip(preds, 0, 10)))


# 2) RMSE — popularity baseline
def compute_rmse_baseline(test_data):
    actuals = test_data['rating'].values
    preds = np.array([movie_avg_dict.get(m, global_avg) for m in test_data['movie_id'].values])
    return np.sqrt(mean_squared_error(actuals, preds))


# 3) Precision@K
def precision_at_k(test_data, rec_func, k=3, threshold=7):
    precisions = []
    for uid in test_data['user_id'].unique():
        relevant = set(test_data[(test_data['user_id'] == uid) & (test_data['rating'] >= threshold)]['movie_id'])
        if not relevant:
            continue
        recs = rec_func(uid, n=k)
        if recs.empty:
            precisions.append(0.0)
            continue
        hits = len(set(recs['movie_id']) & relevant)
        precisions.append(hits / k)
    return np.mean(precisions) if precisions else 0.0


# 4) Baseline recommend functie
def baseline_recommend(user_id, n=3):
    rated = watched_movies.get(user_id, set())
    top = movie_stats[~movie_stats['movie_id'].isin(rated)].head(n)
    rows = []
    for rank, (_, r) in enumerate(top.iterrows(), 1):
        g = ', '.join(r['genres']) if isinstance(r['genres'], list) else str(r['genres'])
        rows.append({'rank': rank, 'movie_id': r['movie_id'],
                     'title': r['title'], 'genres': g,
                     'score': round(r['popularity_score'], 3)})
    return pd.DataFrame(rows)


# 5) Evaluatie uitvoeren
print("RMSE berekenen op volledige testset …")
rmse_hybrid = compute_rmse_hybrid(test_df)
rmse_baseline = compute_rmse_baseline(test_df)

# Precision@3 op een steekproef van testgebruikers
sample_size = min(500, test_df['user_id'].nunique())
sample_users = np.random.choice(test_df['user_id'].unique(), size=sample_size, replace=False)
sample_test_df = test_df[test_df['user_id'].isin(sample_users)]

print(f"Precision@3 berekenen op {sample_size} testgebruikers …")
p3_hybrid = precision_at_k(sample_test_df, recommend, k=3, threshold=7)
p3_baseline = precision_at_k(sample_test_df, baseline_recommend, k=3, threshold=7)

# 6) Resultaten tonen
print(f"\n{'Metric':<20} {'Hybrid':>10} {'Baseline':>10}")
print(f"{'-'*42}")
print(f"{'RMSE':<20} {rmse_hybrid:>10.4f} {rmse_baseline:>10.4f}")
print(f"{'Precision@3':<20} {p3_hybrid:>10.4f} {p3_baseline:>10.4f}")

# 7) 5-fold cross-validation (SVD)
print("\n5-Fold Cross-Validation (SVD)")
print("-" * 42)
kf = SurpriseKFold(n_splits=5, random_state=42, shuffle=True)
cv_rmses = []
for fold_i, (cv_trainset, cv_testset) in enumerate(kf.split(surprise_data), 1):
    cv_svd = SVD(
        n_factors=best_params['n_factors'],
        n_epochs=best_params['n_epochs'],
        lr_all=best_params['lr_all'],
        reg_all=best_params['reg_all'],
        random_state=42
    )
    cv_svd.fit(cv_trainset)
    predictions = cv_svd.test(cv_testset)
    fold_rmse = accuracy.rmse(predictions, verbose=False)
    cv_rmses.append(fold_rmse)
    print(f"  Fold {fold_i}: RMSE = {fold_rmse:.4f}")

print(f"\n  Mean CV RMSE: {np.mean(cv_rmses):.4f} ± {np.std(cv_rmses):.4f}")

Computing RMSE on the full test set …
Computing Precision@3 on 500 sampled test users …

Metric                   Hybrid   Baseline
------------------------------------------
RMSE                     1.7146     1.7146
Precision@3              0.0448     0.0475

── 5-Fold Cross-Validation (SVD) ───────────────────────────────────────
  Fold 1: RMSE = 1.6018
  Fold 2: RMSE = 1.6257
  Fold 3: RMSE = 1.6010
  Fold 4: RMSE = 1.5971
  Fold 5: RMSE = 1.5906

  Mean CV RMSE: 1.6032 ± 0.0119


## Voorbeeldaanbevelingen

Top-3 aanbevelingen voor enkele gebruikers met verschillende activiteitsniveaus, inclusief de hybride score en genres.

In [56]:
# Kies 5 gebruikers met verschillende activiteitsniveaus
active_users = train_df['user_id'].value_counts()
example_users = list(active_users.index[:3])          # 3 meest actieve users
least_active = active_users.tail(2).index.tolist()    # 2 minst actieve users
example_users.extend(least_active)

for uid in example_users:
    n_rated = ratings_per_user.get(uid, 0)
    tag = "  ← cold-start (popularity)" if uid in cold_users else ""
    print(f"\n{'='*72}")
    print(f"  User {uid}   ({n_rated} training ratings){tag}")
    print(f"{'='*72}")
    recs = recommend(uid, n=3)
    for _, row in recs.iterrows():
        print(f"  #{int(row['rank'])}  {row['title']:<50s}  score={row['score']:.3f}")
        print(f"      Genres: {row['genres']}")


  User 3   (5 training ratings)  ← cold-start (popularity)
  #1  Interstellar                                        score=57.068
      Genres: Adventure, Drama, Sci-Fi
  #2  Now You See Me                                      score=49.237
      Genres: Crime, Mystery, Thriller
  #3  This Is the End                                     score=39.873
      Genres: Comedy, Fantasy

  User 4   (5 training ratings)  ← cold-start (popularity)
  #1  1917                                                score=58.907
      Genres: Drama, War
  #2  Star Trek Into Darkness                             score=51.827
      Genres: Action, Adventure, Sci-Fi
  #3  Now You See Me                                      score=49.237
      Genres: Crime, Mystery, Thriller

  User 5   (5 training ratings)  ← cold-start (popularity)
  #1  1917                                                score=58.907
      Genres: Drama, War
  #2  Star Trek Into Darkness                             score=51.827
      Genres: A

## Resultaten en discussie

### Evaluatie van de requirements

Het systeem voldoet grotendeels aan de gedefinieerde modelvereisten. Het genereert voor elke gebruiker drie film-aanbevelingen op basis van een hybride architectuur die collaborative filtering (SVD) en content-based filtering (genre cosine similarity) combineert. Het SVD-model modelleert gebruikers- en itembias, en het content-based component gebruikt one-hot genrevectoren.

Voor cold-start gebruikers (< 5 training ratings) is een populariteitsbaseline beschikbaar met afgedwongen genrediversiteit. In de praktijk valt vrijwel niemand in deze categorie doordat gebruikers met < 5 totale ratings al zijn gefilterd en het given-5 protocol iedereen minimaal 5 training ratings geeft.

De hybride score wordt berekend volgens de vooraf gedefinieerde formule (0.8 × collab + 0.2 × content). De derde aanbeveling wordt uit een ander genrecluster gekozen voor diversiteit.

Van de **tien requirements worden er acht volledig gerealiseerd**. Twee worden slechts gedeeltelijk bereikt: de prestatie-eis dat het hybride model beter presteert dan de populariteitsbaseline wordt nog niet gehaald.

### Modelprestaties

| Metric | Hybrid | Baseline |
|------|------|------|
| RMSE | 1.7146 | 1.7146 |
| Precision@3 | 0.0448 | 0.0475 |

Het hybride model behaalt **geen lagere RMSE** en een **iets lagere Precision@3** dan de baseline. De 5-fold cross-validation van het SVD-model toont een stabiele RMSE van **1.6032 ± 0.0119**, wat aangeeft dat het collaborative filtering component zelf wel consistente voorspellingen leert.

### Interpretatie

Het hybride model functioneert technisch zoals ontworpen, maar levert in deze configuratie nog geen prestatieverbetering ten opzichte van de baseline. Dit is een bekend verschijnsel: een populariteitsmodel is vaak een sterke baseline omdat populaire items veel interacties hebben.

De toegevoegde waarde ligt momenteel in **personalisatie en diversiteit**: het systeem modelleert individuele voorkeuren en genereert gevarieerde aanbevelingen, terwijl de baseline uitsluitend populaire films aanbeveelt.

### Mogelijke verbeteringen

- **Rijkere content features:** naast genres ook acteurs, regisseurs, plotbeschrijvingen of tekst-embeddings gebruiken.
- **Dynamische hybride gewichten:** de balans tussen collab en content automatisch leren in plaats van handmatig instellen.
- **Aanvullende metrics:** Recall@k, NDCG, of online A/B-testing voor een realistischere evaluatie.
- **Meer trainingsdata per gebruiker:** een hoger given-n (bijv. 10) zodat het SVD-model meer interacties per gebruiker heeft om van te leren.

## Bronnen

[1] K. Falk, Practical Recommender Systems. Manning Publications, 2019.